# Stage 1 — The portfolio you built

**Checkpoint:** `stage-1`  ·  run `python verify.py` first — it should say *Stage 1 verified*.

You generated 12,000 small businesses that never existed. This notebook is the visual version
of the audit on your lab sheet — but it goes one step further than the sheet does.

**Two jobs here.** The first is the audit: are the counts right, is anything null, does the data
**rank-order** the way credit intuition says it should? The second is the portrait: *what kind of
book is this?* Who got in, who didn't, and which of these columns actually carries signal.

You leave with a **checkpoint card** (section 10) and three pieces of **homework** that walk
straight into Session 2's first 45 minutes. Session 2 asks you to bin features and defend
excluding some of them. Everything after section 6 exists to make that a decision you have
evidence for, rather than a guess.

In [ ]:
# --- bootstrap: find the repo root, make the repo importable ---------------
# The modules use paths relative to the repo root (e.g. Path("score/models")),
# so we chdir there. This works whether you launched Jupyter from the repo
# root or from notebooks/.
import os, sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "verify.py").exists()), None)
assert root is not None, "Could not find the repo root (no verify.py above cwd)."
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110})

print(f"repo root: {root}")
print(f"stage.txt: {(root / 'stage.txt').read_text().strip()}")

In [ ]:
# --- stage guard: fail loudly and usefully, not mysteriously ---------------
NEEDS = [
    "shared/data/raw/businesses.parquet",
    "shared/data/raw/portfolio.parquet",
    "shared/data/raw/panel.parquet",
]
missing = [p for p in NEEDS if not Path(p).exists()]
if missing:
    raise SystemExit(
        "This notebook needs stage-1 artifacts. Missing:\n  "
        + "\n  ".join(missing)
        + "\n\nYou are at stage " + (Path("stage.txt").read_text().strip())
        + ". Fix with either:\n"
        "  git checkout stage-1      # jump to the finished stage, or\n"
        "  make <the stage's build step>  # build it yourself (see the lab sheet)"
    )
print("stage-1 artifacts present.")

## 1. Row counts — do they match the lab sheet exactly?

In [ ]:
from shared.config import RAW

businesses = pd.read_parquet(RAW / "businesses.parquet")
portfolio  = pd.read_parquet(RAW / "portfolio.parquet")
panel      = pd.read_parquet(RAW / "panel.parquet")

counts = pd.DataFrame(
    [{"file": "businesses.parquet", "rows": len(businesses), "columns": businesses.shape[1]},
     {"file": "portfolio.parquet",  "rows": len(portfolio),  "columns": portfolio.shape[1]},
     {"file": "panel.parquet",      "rows": len(panel),      "columns": panel.shape[1]}]
).set_index("file")

expected = {"businesses.parquet": 12000, "portfolio.parquet": 8336, "panel.parquet": 200064}
counts["expected"] = pd.Series(expected)
counts["match"] = np.where(counts["rows"] == counts["expected"], "OK", "MISMATCH")
counts

> **Why is 8,336 not a round number?** It is the *booked* subset — the applicants who were
> approved **and** took the money. It falls out of the process rather than being chosen, which
> is exactly how a real book accumulates. Section 3 shows you *who* that filtering kept.

## 2. Who is actually in this book?

Counts tell you the data arrived. They do not tell you what you are lending to. Before any
modelling, a credit officer wants the shape of the book: what kinds of businesses, how big,
how established.

In [ ]:
SEGMENTS = ["industry", "region", "entity_type", "loan_purpose"]

mix = pd.DataFrame([
    {"levels":      businesses[c].nunique(),
     "most common": businesses[c].value_counts().idxmax(),
     "top share":   businesses[c].value_counts(normalize=True).max(),
     "rarest":      businesses[c].value_counts().idxmin(),
     "rare share":  businesses[c].value_counts(normalize=True).min()}
    for c in SEGMENTS], index=SEGMENTS)
mix.index.name = "field"

display(mix.style.format({"top share": "{:.1%}", "rare share": "{:.1%}"}))

size = (businesses[["annual_revenue", "net_income", "employees",
                    "years_in_business", "requested_amount"]]
        .describe(percentiles=[.10, .50, .90]).T[["min", "10%", "50%", "90%", "max", "mean"]])
size.style.format("{:,.0f}")

In [ ]:
rev_k = businesses["annual_revenue"] / 1e3

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(rev_k.clip(upper=3000), bins=60, color="#5b8ac6")
ax.axvline(rev_k.median(), color="#b3372e", ls="--", lw=1.5,
           label=f"median ${rev_k.median():,.0f}k")
ax.axvline(rev_k.mean(), color="#333333", ls=":", lw=1.8,
           label=f"mean ${rev_k.mean():,.0f}k")
ax.set_xlabel("annual revenue ($k, clipped at $3M for display)")
ax.set_title("Small-business revenue is long-tailed — the mean is not a typical borrower")
ax.legend(); plt.tight_layout(); plt.show()

print(f"largest borrower: ${businesses['annual_revenue'].max()/1e6:,.1f}M revenue "
      f"— {businesses['annual_revenue'].max() / businesses['annual_revenue'].median():,.0f}x the median")

**Checkpoint.** The mean revenue is roughly 1.7x the median and the largest borrower is ~57x it.
That skew is the reason Session 2 **bins** features into a scorecard instead of fitting raw
values: one $25M outlier should not be allowed to bend the curve for 12,000 corner shops.

## 3. The booking funnel — the book is not the applicant pool

12,000 applied. 8,336 ended up on the book. Something filtered the other 3,664, and that filter
was not random — it was a credit decision.

Here is the part you will never get in real life: this dataset defines `default` for **everyone**,
including the applicants who were declined or walked away. Their default is a *counterfactual* —
what would have happened had you lent to them. On a real book those rows are blank forever, which
is why selection bias is hard to see and easy to deny. Look at it once, here, where it is visible.

In [ ]:
def profile(d, label):
    return pd.Series({"n": len(d),
                      "default_rate":   d["default"].mean(),
                      "dscr":           d["dscr"].mean(),
                      "utilization":    d["utilization"].mean(),
                      "leverage":       d["leverage"].mean(),
                      "annual_revenue": d["annual_revenue"].mean()}, name=label)

booked, declined = businesses[businesses["booked"] == 1], businesses[businesses["booked"] == 0]
funnel = pd.DataFrame([profile(businesses, "all applicants"),
                       profile(booked,     "booked (the book)"),
                       profile(declined,   "not booked")])

display(funnel.style.format({"n": "{:,.0f}", "default_rate": "{:.1%}", "dscr": "{:.2f}",
                             "utilization": "{:.3f}", "leverage": "{:.2f}",
                             "annual_revenue": "${:,.0f}"}))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
vals = funnel["default_rate"] * 100
axes[0].bar(range(3), vals, color=["#8a8f98", "#2f7a4f", "#b3372e"])
axes[0].set_xticks(range(3))
axes[0].set_xticklabels(["all\napplicants", "booked\n(the book)", "not\nbooked"])
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.5, f"{v:.1f}%", ha="center")
axes[0].set_ylabel("default rate (%)"); axes[0].set_ylim(0, vals.max() * 1.2)
axes[0].set_title("The book is the safe half of who applied")

bins = np.linspace(businesses["dscr"].quantile(.01), businesses["dscr"].quantile(.99), 40)
axes[1].hist(booked["dscr"],   bins=bins, alpha=.6, density=True, color="#2f7a4f", label="booked")
axes[1].hist(declined["dscr"], bins=bins, alpha=.6, density=True, color="#b3372e", label="not booked")
axes[1].set_xlabel("dscr"); axes[1].set_title("DSCR — what the filter selected on")
axes[1].legend()
plt.tight_layout(); plt.show()

lift = funnel.loc["not booked", "default_rate"] / funnel.loc["booked (the book)", "default_rate"]
print(f"Applicants who never reached the book default {lift:.1f}x as often as those who did.")
print("On a real portfolio you can only ever measure the middle row. This table does not exist.")

**Checkpoint.** Three numbers to carry into Session 2: the applicant pool defaults at **16.7%**,
the book at **10.9%**, the rejected at **29.9%**. A model trained on the book alone learns the
safe half and will look better than it is on new applicants. This is *reject inference*, and it
is the question a validator asks about every through-the-door scorecard.

Session 2 trains on **all 12,000 applicants** — a synthetic-data luxury that buys you an honest
model. Note it in your decision memo: on real data you would not have had that choice.

## 4. Target rates — and what each one is a rate *of*

Four numbers, three different denominators. Getting these confused is one of the most common
ways a model report ends up quietly wrong, so the table names the population every time.

In [ ]:
n_app, n_book = len(businesses), len(portfolio)

rates = pd.DataFrame(
    [("default",             f"all applicants ({n_app:,})",  businesses["default"].mean()),
     ("booked",              f"all applicants ({n_app:,})",  businesses["booked"].mean()),
     ("default",             f"booked only ({n_book:,})",    portfolio["default"].mean()),
     ("deterioration 6-12mo", f"on-book ({n_book:,})",       portfolio["deterioration_next_6_12mo"].mean()),
     ("line_increase_good",  f"on-book ({n_book:,})",        portfolio["line_increase_good"].mean())],
    columns=["rate of", "population", "value"]
).set_index(["rate of", "population"])

rates.style.format({"value": "{:.1%}"})

## 5. Null audit — synthetic data is complete by construction, so a null is a bug

In [ ]:
nulls = pd.Series({
    "businesses": int(businesses.isna().sum().sum()),
    "portfolio":  int(portfolio.isna().sum().sum()),
    "panel":      int(panel.isna().sum().sum()),
}, name="null cells")
print(nulls.to_string())
print("\nPASS — no nulls anywhere." if nulls.sum() == 0 else "\nFAIL — investigate.")

## 6. Default rate by industry — does any segment look implausible?

Read the chart, form an opinion, then **hold it**. Section 7 tests that opinion with a number,
and the two do not agree.

In [ ]:
by_ind = (businesses.groupby("industry")["default"]
          .agg(["size", "mean"]).sort_values("mean", ascending=False))
by_ind.columns = ["applicants", "default_rate"]

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.barh(by_ind.index, by_ind["default_rate"] * 100, color="#5b8ac6")
ax.axvline(businesses["default"].mean() * 100, color="#b3372e", ls="--", lw=1.5,
           label=f"portfolio average {businesses['default'].mean():.1%}")
ax.invert_yaxis(); ax.set_xlabel("default rate (%)"); ax.legend()
ax.set_title("Default rate by industry")
plt.tight_layout(); plt.show()

by_ind.style.format({"default_rate": "{:.1%}"})

## 7. What actually carries signal

Two questions, in order.

**First, does it rank-order?** Credit intuition says: **higher DSCR → safer**,
**higher utilization → riskier.** If the data disagrees, either the intuition or the generator
is wrong. This is the check a validator runs before looking at any model.

**Second, how much is each feature worth?** A feature can rank-order beautifully and still be
worth almost nothing. **Information Value (IV)** puts a number on it: bin the feature, compare
the share of goods to the share of bads in each bin, and add up the differences. The industry
rules of thumb — *< 0.02 unusable, 0.02–0.1 weak, 0.1–0.3 medium, > 0.3 suspiciously strong* —
are conventions, not laws, but they are the conventions your validator will quote back at you.

You will compute WoE by hand in Session 2. This is the aerial view that tells you which features
are worth the 45 minutes.

In [ ]:
def decile_rate(df, col, target="default", q=10, ascending_expect="down"):
    d = df[[col, target]].copy()
    d["bucket"] = pd.qcut(d[col], q=q, duplicates="drop")
    out = d.groupby("bucket", observed=True)[target].agg(["size", "mean"])
    out.columns = ["n", "default_rate"]
    return out

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (col, expect) in zip(axes, [("dscr", "falls"), ("utilization", "rises")]):
    t = decile_rate(businesses, col)
    ax.plot(range(len(t)), t["default_rate"] * 100, marker="o", color="#245bb2")
    ax.set_title(f"{col} deciles — default rate should {expect}")
    ax.set_xlabel(f"{col} decile (low → high)"); ax.set_ylabel("default rate (%)")
plt.tight_layout(); plt.show()

dscr_t = decile_rate(businesses, "dscr")
util_t = decile_rate(businesses, "utilization")
print(f"dscr:        first decile {dscr_t['default_rate'].iloc[0]:.1%} -> last {dscr_t['default_rate'].iloc[-1]:.1%}  ({'OK, falls' if dscr_t['default_rate'].iloc[0] > dscr_t['default_rate'].iloc[-1] else 'UNEXPECTED'})")
print(f"utilization: first decile {util_t['default_rate'].iloc[0]:.1%} -> last {util_t['default_rate'].iloc[-1]:.1%}  ({'OK, rises' if util_t['default_rate'].iloc[-1] > util_t['default_rate'].iloc[0] else 'UNEXPECTED'})")

In [ ]:
from shared.config import LEAKAGE_COLUMNS

CANDIDATES = [c for c in businesses.columns
              if c not in LEAKAGE_COLUMNS and c != "business_id"]


def iv_and_shape(s, y, q=5):
    """Information Value, plus: does the bad rate move in one direction across bins?"""
    if pd.api.types.is_numeric_dtype(s):
        bucket = pd.qcut(s, q, duplicates="drop") if s.nunique() > 10 else s
        ordered = True
    else:
        bucket, ordered = s.astype(str), False

    t = pd.crosstab(bucket, y)
    if t.shape[1] < 2 or len(t) < 2:
        return np.nan, "n/a"

    goods = t.iloc[:, 0] / t.iloc[:, 0].sum()
    bads  = t.iloc[:, 1] / t.iloc[:, 1].sum()
    iv = float(((goods - bads) * np.log((goods + 1e-6) / (bads + 1e-6))).sum())

    if not ordered:
        return iv, "n/a (categorical)"
    rate = t.iloc[:, 1] / t.sum(axis=1)
    return iv, ("yes" if rate.is_monotonic_increasing or rate.is_monotonic_decreasing else "no")


def rule_of_thumb(v):
    if v < 0.02:  return "unusable"
    if v < 0.10:  return "weak"
    if v < 0.30:  return "medium"
    return "strong — check for leakage"


scan = pd.DataFrame([(c, *iv_and_shape(businesses[c], businesses["default"])) for c in CANDIDATES],
                    columns=["feature", "IV", "monotonic"]).set_index("feature")
scan = scan.sort_values("IV", ascending=False)
scan["verdict"] = scan["IV"].map(rule_of_thumb)

print(f"{len(CANDIDATES)} candidate features (everything except the deny-list and the id).")
print(f"{(scan['IV'] < 0.02).sum()} of them are below the 0.02 'unusable' line.\n")
scan.style.format({"IV": "{:.3f}"}).background_gradient(subset=["IV"], cmap="Blues")

### The chart in section 6 was not a finding

Go back and look at the industry bar chart. Construction defaults at ~18.4%, Wholesale at ~15.1%
— a spread of about 3 points, with a tidy ordering and a plausible story attached to it
(construction is cyclical, wholesale is not). It looks like something.

Its IV is **0.006** — three times below the "unusable" line. And you can do better than
"probably noise": open `shared/data_generator.py` and read the `logit = ...` block. `industry`
is not in it. Neither is `region`, `entity_type`, `loan_purpose`, `term_months`,
`collateral_flag`, `trade_lines` or `employees`. Their true effect on default is **exactly zero
by construction**; the 3-point spread is the residue of ten buckets of ~1,200 draws.

Nobody was fooled by bad data. The chart was honest. The *story* — construction is cyclical,
wholesale is not — was supplied by the reader, and it fit.

**This is the lesson of the section.** A chart shows you a difference; IV tells you whether the
difference is worth anything. Financial health and track record — `dscr`, `leverage`,
`utilization`, `prior_delinquencies`, `years_in_business` — carry essentially all the signal in
this portfolio. What sector you are in and where you are carry almost none.

Two things worth noticing before you decide anything:

- **`prior_delinquencies` has high IV (0.19) and is flagged non-monotonic.** Look at why:
  the bad rate climbs 12% → 20% → 28% → 42% and then *dips* to 40% at four delinquencies —
  on **35 accounts**. Is that a real reversal or an empty cell? Your answer determines how you
  bin it in Session 2. (This is homework question 1.)
- **`monotonic = n/a (categorical)`** is not a pass mark. Categoricals have no natural order, so
  the test does not apply — which is exactly why a weak categorical is so easy to keep by accident.

### The opposite trap: signal with no cause

Zero-IV features are the easy case — you drop them and nothing is lost. The dangerous case is
the reverse: a feature with **real, stable, medium-strength IV and no causal role at all.**

The scan has one. Go back and read the `logit = ...` block again, then list every candidate with
usable IV that does *not* appear in it.

In [ ]:
# Every column the DGP's default logit actually touches (shared/data_generator.py).
# profit_margin is in there too, but it is not a raw column — it is net_income / annual_revenue.
DGP_DRIVERS = ["dscr", "leverage", "utilization", "prior_delinquencies", "years_in_business",
               "annual_revenue", "credit_history_months", "current_ratio", "public_records"]

audit = scan.copy()
audit["in the logit?"] = np.where(audit.index.isin(DGP_DRIVERS), "yes", "NO")
suspects = audit[(audit["in the logit?"] == "NO") & (audit["IV"] >= 0.02)]

display(audit[["IV", "monotonic", "in the logit?"]].head(11).style.format({"IV": "{:.3f}"}))

print("Usable IV, but no causal role in the DGP:\n")
for c in suspects.index:
    # Spearman, not Pearson: IV is computed on bins, so rank agreement is what transfers.
    r = businesses[c].corr(businesses["annual_revenue"], method="spearman")
    print(f"  {c:18s} IV {suspects.loc[c, 'IV']:.3f}   rank-corr with annual_revenue {r:+.2f}"
          f"   <- borrowed from here")

**`requested_amount` is the one to remember.** IV 0.103, cleanly monotonic, medium strength — on
the scan it looks like a keeper. It appears nowhere in the default logit. Its entire signal is
borrowed from `annual_revenue`, which *is* a driver, because the generator builds it as

```python
requested_amount = annual_revenue * uniform(0.05, 0.50)
```

Rank-correlation 0.85 with revenue. The feature is not lying to you — the IV is real and it will
hold up on your validation split, on a holdout, and next quarter. It is real for exactly as long
as the relationship between what a business earns and what it asks for stays put.

**Why that is worse than a weak feature.** Change the product — a campaign pushing larger asks, a
new minimum ticket, a segment that borrows differently — and the borrowed signal drains away
while every drift monitor you have stays green. `requested_amount`'s own distribution has not
moved. Its *relationship to the thing that actually drives risk* has, and nothing is watching that.

`net_income` is the same story, one step milder (IV 0.100, rank-corr 0.65 with revenue).

Session 2 drops both and keeps `annual_revenue`. That is a defensible call — take the driver,
not the proxy — but it was **a call**, not an output. On real data you cannot open the generator
and check. You would have only the correlation, the domain argument, and your judgement, and the
question a validator asks is the same one either way:

> *If this feature works, what is the mechanism? And what would have to change for it to stop?*

You get to practise that question here with the answer key available. That is the entire reason
this data is synthetic.

## 8. The behavioural panel — 24 months of it feeds Session 4

In [ ]:
monthly = panel.groupby("month_index").agg(
    utilization=("utilization", "mean"),
    days_past_due=("days_past_due", "mean"),
    deposit_inflow=("deposit_inflow", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col, title in zip(axes, ["utilization", "days_past_due", "deposit_inflow"],
                          ["mean utilization", "mean days past due", "mean deposit inflow"]):
    ax.plot(monthly["month_index"], monthly[col], color="#245bb2")
    ax.set_title(title); ax.set_xlabel("month")
plt.tight_layout(); plt.show()

print(f"{panel['business_id'].nunique():,} businesses x {panel['month_index'].nunique()} months = {len(panel):,} rows")

## 9. The deny-list — six columns no model may ever see

A deny-list in someone's head is not a control. A deny-list in `config.py` is.

In [ ]:
from shared.config import LEAKAGE_COLUMNS

why = {
    "pd_default_origination":     "the data generator's ground-truth PD — the answer, laundered",
    "default":                    "the answer itself",
    "risk_based_rate":            "priced FROM the true PD, so it leaks it",
    "booked":                     "a post-decision outcome; unknown at decision time",
    "deterioration_next_6_12mo":  "Session 4's target — future information",
    "line_increase_good":         "Session 4's other target — future information",
}
pd.DataFrame({"column": LEAKAGE_COLUMNS,
              "present in businesses?": [c in businesses.columns for c in LEAKAGE_COLUMNS],
              "why it is forbidden": [why.get(c, "") for c in LEAKAGE_COLUMNS]}).set_index("column")

In [ ]:
# The 15 features Session 2 actually trains on. Read from the real module when it exists,
# so this notebook can never drift from the platform.
try:
    from score.src.feature_engineering import FEATURE_COLUMNS as S2_FEATURES
    source = "score/src/feature_engineering.py"
except ModuleNotFoundError:
    S2_FEATURES = ["years_in_business", "annual_revenue", "dscr", "current_ratio", "leverage",
                   "utilization", "credit_history_months", "prior_delinquencies", "trade_lines",
                   "public_records", "debt_to_income", "revenue_per_employee", "profit_margin",
                   "industry", "entity_type"]
    source = "hard-coded here — score/ does not exist until stage-2"

SAFE    = CANDIDATES
kept    = [c for c in S2_FEATURES if c in SAFE]
dropped = [c for c in SAFE if c not in S2_FEATURES]
derived = [c for c in S2_FEATURES if c not in businesses.columns]

print(f"safe columns at stage-1 : {len(SAFE):>2}")
print(f"features S2 trains on   : {len(S2_FEATURES):>2}   (source: {source})\n")
print(f"  kept as-is   ({len(kept):>2}): " + ", ".join(kept))
print(f"  dropped      ({len(dropped):>2}): " + ", ".join(dropped))
print(f"  built new    ({len(derived):>2}): " + ", ".join(derived))

### Ratios, not levels

Session 2 does not feed the scorecard raw columns. It drops eight of them and builds three new
ones out of the pieces:

| built | from | why |
|---|---|---|
| `debt_to_income` | `total_debt` / `net_income` | $2M of debt means nothing until you know what it services |
| `revenue_per_employee` | `annual_revenue` / `employees` | productivity, not size |
| `profit_margin` | `net_income` / `annual_revenue` | quality of the revenue |

Does it work? Compare each ratio's IV with the IV of the raw columns it was built from.

In [ ]:
# Same three ratios Session 2 builds (score/src/feature_engineering.py::compute_features).
ratios = pd.DataFrame({
    "debt_to_income":       (businesses["total_debt"] / businesses["net_income"].abs().clip(lower=1)).clip(0, 50),
    "revenue_per_employee":  businesses["annual_revenue"] / businesses["employees"].clip(lower=1),
    "profit_margin":        (businesses["net_income"] / businesses["annual_revenue"].clip(lower=1)).clip(-1, 1),
})

PARENTS = {"debt_to_income":       ["total_debt", "net_income"],
           "revenue_per_employee": ["annual_revenue", "employees"],
           "profit_margin":        ["net_income", "annual_revenue"]}

rows = []
for ratio, parents in PARENTS.items():
    iv, mono = iv_and_shape(ratios[ratio], businesses["default"])
    rows.append({"built feature": ratio, "IV": iv, "monotonic": mono,
                 "built from":    " / ".join(parents),
                 "their IVs":     " / ".join(f"{scan.loc[p, 'IV']:.3f}" for p in parents),
                 "beats both?":   "yes" if all(iv > scan.loc[p, "IV"] for p in parents) else "no"})

ratio_scan = pd.DataFrame(rows).set_index("built feature")
ratio_scan.style.format({"IV": "{:.3f}"})

**One clear win, and two features that univariate IV says you should not have built.**

`debt_to_income` is the win: IV 0.120, against 0.014 for `total_debt` on its own. Two million of
debt tells you nothing until you divide it by what services it — the ratio created signal that
neither parent had. It beats both parents, and it is the only one here that does.

The other two lose. `revenue_per_employee` (0.081) is worth *less* than plain `annual_revenue`
(0.142); dividing threw away the size information that was doing the work. `profit_margin`
(0.006) lands below the unusable line while its parent `net_income` sits at 0.100.

**So is Session 2's feature engineering wrong?** Not necessarily — and this is the point of the
section. Univariate IV scores one feature at a time, in isolation. It cannot see:

- **Correlation.** `annual_revenue`, `net_income` and `total_debt` all move together. A
  scorecard fed all three double-counts size. One ratio may be worth less alone and more
  alongside the rest.
- **Stability.** A ratio is often steadier across time and segments than the levels it came
  from — and a scorecard has to survive next year, not just this sample.
- **Explainability.** "Your debt is high relative to your income" is a reason code you can put
  in a decline letter. "Your `total_debt` is 0.4 standard deviations above the mean" is not.

**Use IV as a screen, not a verdict.** It tells you where to look and what to ask about. Whether
`profit_margin` earns its place in the scorecard is a question for Session 2 — bring it.

### Feel the poison — three rungs, 20 seconds

Train the same model three times: on honest features, then with one denied column added,
then with the answer itself. Read the three numbers before you read the explanation under them.


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

HONEST = ["dscr", "leverage", "utilization", "prior_delinquencies"]
X_ok, y = businesses[HONEST], businesses["default"]

rungs = [
    ("honest — 4 real features", X_ok,                                              ""),
    ("+ pd_default_origination", X_ok.assign(cheat=businesses["pd_default_origination"]),
                                                        "the generator's true PD"),
    ("+ default",                X_ok.assign(cheat=businesses["default"]),
                                                        "the answer itself"),
]

for name, X, note in rungs:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
    auc = roc_auc_score(yte, LGBMClassifier(verbose=-1).fit(Xtr, ytr).predict_proba(Xte)[:, 1])
    print(f"{name:<26} AUC = {auc:.4f}   {note}")


**The 1.0 is not the lesson.** Training on `default` is the answer copied into the features —
obviously broken, trivially caught, and nobody ships it.

**The middle rung is the one that gets you.** `pd_default_origination` buys about **+0.10 AUC**
over the honest model. It is not suspicious-looking. It does not scream. A model with that number
sails through review, ships, and then performs at 0.73 in production — because at decision time
the true PD does not exist. That is what leakage actually looks like in a bank: not a 1.0, but a
number good enough to be believed.

Now delete anything you built here. The deny-list in `shared/config.py` is the control; this cell
is the demonstration of what it is protecting you from.


## 10. Your checkpoint card

Five things you now know about this portfolio. Everything below is recomputed from the data on
every run — nothing is typed in — so if you regenerate the book, the card follows.

In [ ]:
n_app, n_book = len(businesses), int(businesses["booked"].sum())
ratio_cols = list(ratios.columns)
strong = scan[scan["IV"] >= 0.10].index.tolist()
weak   = scan[scan["IV"] <  0.02].index.tolist()

print(f"""
CHECKPOINT CARD — the small-business book you built
{'=' * 66}

1. SIZE       {n_app:,} applicants -> {n_book:,} on the book ({n_book / n_app:.0%} take-up),
              plus {panel['month_index'].nunique()} months of behaviour on {panel['business_id'].nunique():,} accounts.

2. RISK       {businesses['default'].mean():.1%} of applicants default.
              The book defaults at {portfolio['default'].mean():.1%}; those who never reached it, at {businesses.loc[businesses['booked'] == 0, 'default'].mean():.1%}.
              The book is the safe half. Never quote one of these rates without its population.

3. SHAPE      Median revenue ${businesses['annual_revenue'].median():,.0f}, mean ${businesses['annual_revenue'].mean():,.0f}, max ${businesses['annual_revenue'].max():,.0f}.
              Long-tailed. This is why Session 2 bins instead of fitting raw values.

4. SIGNAL     {len(strong)} features reach medium strength (IV >= 0.10): {', '.join(strong)}.
              {len(weak)} are unusable (IV < 0.02), including industry and region.
              Financial health, track record and size carry this book. Sector and region do not.

5. LIMITS     {len(LEAKAGE_COLUMNS)} columns are permanently off the table ({', '.join(LEAKAGE_COLUMNS)}).
              Session 2 trains on {len(S2_FEATURES)} features: {len(kept)} raw + {len(ratio_cols)} built from the dropped ones.

{'=' * 66}
Session 2 gates the scorecard at AUC >= 0.78. You now know what it has to work with.
""")

---

## Homework — three things, before Session 2

Session 2 opens with 45 minutes of manual binning (Lab 2, Part A) and asks you to defend three
decisions. Arriving cold costs you most of that time. Arriving with these costs you 30 minutes
tonight and doubles what you get out of the session.

**1. Bin one feature by hand and compute WoE.**
Pick `dscr`, `utilization`, or `prior_delinquencies`. Cut it into 5 bins, and for each bin
compute the count, the bad rate, and the Weight of Evidence:

```
WoE(bin) = ln( share of all GOODS in this bin / share of all BADS in this bin )
```

It is four lines of pandas — write them yourself; the notebook above deliberately does not.
Then answer: **does your bad rate move in one direction across the bins?** If you picked
`prior_delinquencies`, decide what to do with the 35 accounts at four delinquencies — merge them
upward, or trust the reversal? Bring your table and your answer.

**2. Pick five in and five out.**
From the 20 safe columns, name five you would put in a scorecard and five you would refuse.
One line of justification each. IV is evidence, not a verdict — a feature can be strong and still
be unacceptable (unstable, unexplainable to a declined applicant, or a proxy for something you
may not price on). Say so where it applies.

**3. Seal a number.**
Write down, now, the AUC you expect a scorecard built on your five features to reach on held-out
data. One number. Fold it up.

Session 2's gate is **0.78, asserted in code**. The distance between your number and what the
model actually does is the most useful thing you will learn that evening — and you only get to
learn it if you commit to a number *before* you see the answer.

**4. (from the lab sheet)** Read the reference build journal —
[teamyan.substack.com/p/claude-code-credit-app-suite](https://teamyan.substack.com/p/claude-code-credit-app-suite).
Five lines: which of the four decision apps would you trust least, and what would you check first?

---
**Next:** `02_stage2_score_spine.ipynb` — turn this portfolio into a credit score that has to
pass a hard gate, or it does not ship.